# HyperSSM on the FSM benchmark (PD-SSM's Tables 2–3, extended)

**Task-free snapping (this version).** Nothing about a task's group is supplied, and every snap is used the same way
in training and at test:
* **ℝ channels are exactly ±1 in the forward pass** (straight-through, tanh gradient), in training and at test.
  ±1 are the only bounded real transitions, so this is the analogue of PD-SSM's hardmax.
* **ℂ lattices are discovered, not given.** Training is continuous; afterwards each complex layer is snapped to the
  **smallest** n whose n-th roots of unity give the best validation accuracy (n ≤ `LATTICE_MAX`); the readout is then
  fine-tuned on the snapped states, so the evaluated model is the model last trained.
* **ℍ channels are continuous** throughout (as before: `hard` mode never rounded quaternions).
* **S5: no pretrained or frozen sign layer**; both layers are trained jointly.

Each run records the continuous results (before discovery), the discovered n, and the final snapped results.
The previous behaviour (task-given lattices, sign pretraining) is recovered with `DISCOVER_LATTICE = False`,
`REAL_STRAIGHT_THROUGH = False`, `QUOTIENT_FIRST = True`.

**One architecture** (the `hyperssm` package below): channels over a chosen number system (ℝ±, ℂ, ℍ), stacked, each
layer reading the one below through its characters, with a linear readout. **Per task, only the algebra and the depth
change**, as the theory prescribes: parity → ℝ±; cycle navigation → ℂ; even pairs → ℂ on (previous, current) pairs;
modular arithmetic → a dense M₁₆(ℂ) block (its minimal 3-number ℂ→ℂ cascade is checked by hand); A5 → ℍ; S5 → ℝ±→ℍ.

**Protocol (Terzić et al., NeurIPS 2025; official code):** training lengths 3–40, loss on the final token only,
validation on lengths 40–256 (best checkpoint); Table 2 reports mean ± std over 5 seeds, Table 3 the best of 3.
**Extension:** every model is also tested on lengths 256–1000, and S5 is run with all 120 generators.
Baseline numbers are taken from the paper and printed next to ours.

**Status (CPU checks, final-token protocol):** parity, cycle navigation, even pairs, A5,12, S5,4 and S5,32 train to
exact (100% at lengths 40–256 and 256–1000). Modular arithmetic uses a dense M₁₆(ℂ) block (as in Walker et al.'s
best model, which also needs a dense 16×16 block): 100% at 40–256 and 96.9% at 256–1000 after 3k steps (1 seed).
Its minimal form, a 3-number ℂ→ℂ cascade, is exactly representable (hand-set check below) but not yet learned.

Runs on a CPU runtime (most models have tens to a few thousand parameters); modular arithmetic (a 16×16 complex block
per token pair, ~51k parameters) is much faster on a GPU.
Results are appended to `RESULTS_PATH` and skipped on re-run, so the notebook can be resumed.

In [ ]:
import itertools, math, time, json, os, copy, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ========================= configuration =========================
TASKS = ["parity", "even_pairs", "cycle_nav", "mod_arith",
         "A5_2", "A5_6", "A5_8", "A5_12", "S5_4", "S5_8", "S5_32", "S5_120"]
SEEDS = [0, 1, 2, 3, 4]          # Table 2 (FSA): mean ± std over 5 seeds; Table 3 (A5/S5): best of seeds
STEPS = 20_000                   # per attempt (early stopping at 0.9995)
BATCH = 256
EVAL_EVERY = 500
N_VAL, N_EXT = 2048, 256         # validation (lengths 40–256) and extended test (256–1000)
LR = 1e-2
LR_BY_TASK = {"mod_arith": 3e-3}   # the dense M16(ℂ) block trains with a smaller step
CURR_HOLD = 0.3                  # fraction of training on lengths 3–5
RESTARTS, RESTARTS_ABELIAN, RESTARTS_MOD = 8, 20, 0   # mod arith: one long run (the dense block learns slowly but steadily)
ABORT_AT, ABORT_BELOW = 0.5, 0.9
EARLY_STOP = True
RESULTS_PATH = "hyperssm_fsm_results_taskfree.jsonl"   # new file: do not mix with the earlier results

# ---- task-free snapping ----
REAL_STRAIGHT_THROUGH = True     # ℝ channels exactly ±1 in the forward pass (train and test), tanh gradient
DISCOVER_LATTICE = True          # ℂ lattices found from the trained model (smallest best n), not given per task
LATTICE_MAX = 64                 # largest n searched
FINETUNE_STEPS = 2_000           # brief training in the snapped form (straight-through)
FINETUNE_LR_FACTOR = 0.1         # fine-tune learning rate = this × the task's learning rate
N_SEL = 512                      # validation sequences used for checkpoint selection during training
QUOTIENT_FIRST = False           # S5: True restores the pretrained, frozen sign layer   # e.g. "/content/drive/MyDrive/hyperssm/fsm_results.jsonl"

## Tasks
Each generator returns zero-padded tokens `(B, Lmax)`, the final-token targets `(B,)` and lengths `(B,)`.
FSA tasks reproduce `state_tracking_PyTorch/data_dir/fl_tasks/*.py` (same tokens, same targets, same vocabularies).

In [ ]:
def _lengths(B, lmin, lmax, rng): return rng.integers(lmin, lmax + 1, B)

def gen_parity(B, lmin, lmax, rng):
    L = _lengths(B, lmin, lmax, rng); x = rng.integers(1, 3, (B, lmax))
    x[np.arange(lmax)[None] >= L[:, None]] = 0
    y = np.where(((x == 2).sum(1) % 2) == 0, 1, 2)
    return x, y, L

def gen_even_pairs(B, lmin, lmax, rng):
    L = _lengths(B, lmin, lmax, rng); x = rng.integers(1, 3, (B, lmax))
    x[np.arange(lmax)[None] >= L[:, None]] = 0
    y = np.where(x[:, 0] == x[np.arange(B), L - 1], 1, 2)
    return x, y, L

def gen_cycle_nav(B, lmin, lmax, rng):
    L = _lengths(B, lmin, lmax, rng); x = rng.integers(1, 4, (B, lmax))       # 1 STAY, 2 +1, 3 -1
    x[np.arange(lmax)[None] >= L[:, None]] = 0
    pos = (x == 2).sum(1) - (x == 3).sum(1)
    return x, 4 + pos % 5, L

def gen_mod_arith(B, lmin, lmax, rng):
    lmax_even = lmax + (lmax % 2)
    X = np.zeros((B, lmax_even), dtype=np.int64); Y = np.zeros(B, dtype=np.int64); Ls = np.zeros(B, dtype=np.int64)
    for b in range(B):
        n = int(rng.integers(lmin, lmax + 1)); n += n % 2
        res = [0]*n
        for i in range(0, n, 2): res[i] = int(rng.integers(5, 10))
        for i in range(1, n - 1, 2): res[i] = int(rng.integers(1, 4))
        res[-1] = 4
        values, ops = [res[0] - 5], []
        for i in range(1, n - 2, 2):
            op, num = res[i], res[i + 1] - 5
            if op == 3: values[-1] *= num
            else: values.append(num); ops.append(op)
        tot = values[0]
        for j, op in enumerate(ops): tot = tot + values[j + 1] if op == 1 else tot - values[j + 1]
        X[b, :n] = res; Y[b] = tot % 5 + 5; Ls[b] = n
    return X, Y, Ls

# ---- permutation groups ----
def perm_group(name):
    elems = [p for p in itertools.permutations(range(5))
             if name == "S5" or sum(p[i] > p[j] for i in range(5) for j in range(i + 1, 5)) % 2 == 0]
    idx = {p: i for i, p in enumerate(elems)}
    mul = np.array([[idx[tuple(g[h[i]] for i in range(5))] for h in elems] for g in elems])   # mul[g,h] = g∘h
    sign = np.array([(-1) ** sum(p[i] > p[j] for i in range(5) for j in range(i + 1, 5)) for p in elems])
    return elems, idx, mul, sign

GROUPS = {g: perm_group(g) for g in ("A5", "S5")}
STD_GENS = {"A5": [(1, 2, 0, 3, 4), (1, 2, 3, 4, 0)],       # (0 1 2), (0 1 2 3 4)
            "S5": [(1, 0, 2, 3, 4), (1, 2, 3, 4, 0)]}       # (0 1),   (0 1 2 3 4)

def group_generators(task):
    g, n = task.split("_"); n = int(n); elems, idx, mul, sign = GROUPS[g]
    if n >= len(elems): return g, list(range(len(elems)))
    gens = [idx[p] for p in STD_GENS[g]]
    rest = [i for i in range(len(elems)) if i not in gens and i != idx[tuple(range(5))]]
    gens += list(np.random.default_rng(12345 + n).choice(rest, n - 2, replace=False))
    return g, gens

def make_group_gen(task):
    g, gens = group_generators(task); elems, idx, mul, sign = GROUPS[g]; gens = np.array(gens)
    ident = idx[tuple(range(5))]
    def gen(B, lmin, lmax, rng):
        L = _lengths(B, lmin, lmax, rng); tok = rng.integers(1, len(gens) + 1, (B, lmax))
        tok[np.arange(lmax)[None] >= L[:, None]] = 0
        cur = np.full(B, ident); ys = np.zeros((B, lmax), dtype=np.int64)
        for t in range(lmax):
            live = t < L
            cur = np.where(live, mul[gens[np.maximum(tok[:, t] - 1, 0)], cur], cur)   # π_t = g_t ∘ π_{t−1}
            ys[:, t] = cur
        gen.last_all = ys                                                            # per-step targets (SUPERVISION="all")
        return tok, cur, L
    return gen

def task_spec(task):
    """returns (generator, vocab size, number of classes, extra info)"""
    if task == "parity":     return gen_parity, 3, 3, {}
    if task == "even_pairs": return gen_even_pairs, 3, 3, {}
    if task == "cycle_nav":  return gen_cycle_nav, 9, 9, {}
    if task == "mod_arith":  return gen_mod_arith, 10, 10, {}
    g, gens = group_generators(task); elems, idx, mul, sign = GROUPS[g]
    return make_group_gen(task), len(gens) + 1, len(elems), {"group": g, "sign": sign}

## The `hyperssm` package
* `Algebra`: a number system for channels: how a generator becomes a transition, how elements multiply, which
  **characters** χ(q) a layer above or the readout may read, and an optional finite **lattice** to snap to.
* `HyperLayer`: one recurrence, $q_t = r_t\,\exp(v_t)\,q_{t-1} + b_t$ (Definition 2), computed by a parallel scan.
  The generator is **bilinear in the input and in the characters of the layer below at t−1** (Prop. 2), so a layer
  reads the one below exactly the way the readout reads the top: linearly in χ. Optional: a pairwise (kernel-2) input
  for (previous, current) token, selective decay $r_t\in(0,1)$, and an input term $b_t$.
* `HyperSSM`: a stack of layers and a **linear readout on χ** (no MLP after the recurrence: by the capacity rule,
  2K features can fit any labelling of lengths ≤ T once 2K+1 > T, so the readout stays small and linear).
* `hard` mode: every transition and write is projected onto its algebra's lattice (exact ±1, roots of unity), so a
  snapped model is an exact finite-group representation and is correct at every length.
* `straight_through=True`: training uses the projected (exact) values in the forward pass and the continuous
  parameters in the backward pass, so a model cannot fit the data with off-lattice values that a layer above or the
  readout could memorize (needed for modular arithmetic, whose coupling reads the magnitudes of the layer below).

In [ ]:
def prefix_scan(a, mul):
    """Inclusive scan along dim 1: P_t = a_t · … · a_1 (Hillis–Steele, O(log T) parallel steps)."""
    T, d = a.shape[1], 1
    while d < T:
        a = torch.cat([a[:, :d], mul(a[:, d:], a[:, :-d])], 1); d *= 2
    return a

def affine_scan(a, b, mul):
    """h_t = a_t · h_{t-1} + b_t with h_0 = 0, as a parallel scan over affine maps (Eq. 3)."""
    T, d = a.shape[1], 1
    while d < T:
        a, b = (torch.cat([a[:, :d], mul(a[:, d:], a[:, :-d])], 1),
                torch.cat([b[:, :d], mul(a[:, d:], b[:, :-d]) + b[:, d:]], 1))
        d *= 2
    return b

def _qmul(p, q):
    pw, px, py, pz = p.unbind(-1); qw, qx, qy, qz = q.unbind(-1)
    return torch.stack([pw*qw - px*qx - py*qy - pz*qz, pw*qx + px*qw + py*qz - pz*qy,
                        pw*qy - px*qz + py*qw + pz*qx, pw*qz + px*qy - py*qx + pz*qw], -1)

class Algebra:
    name, gdim, sdim = "", 0, 0
    def __init__(self, lattice=None): self.lattice = lattice
    def identity(self, shape, device): raise NotImplementedError
    def transition(self, g, hard): raise NotImplementedError           # unit element from generator g (…, gdim)
    def write(self, w, hard): raise NotImplementedError                # input term b from raw parameters (…, wdim)
    def mul(self, a, b): raise NotImplementedError
    def scale(self, a, r): return a * r[..., None]
    def chi_dim(self, degree): raise NotImplementedError
    def chi(self, q, degree): raise NotImplementedError                 # characters (…, chi_dim)
    wdim = 0

class Real(Algebra):
    """ℝ±: one real number per channel. Transition c = tanh(g) ∈ (−1, 1); lattice: exact ±1 (Z2, a real irrep)."""
    name, gdim, sdim, wdim = "ℝ±", 1, 1, 1
    def identity(self, shape, device): return torch.ones(*shape, 1, device=device)
    def transition(self, g, hard): return torch.sign(g) + (g == 0) if hard else torch.tanh(g)
    def write(self, w, hard): return w.round().clamp(-1, 1) if hard else w
    def mul(self, a, b): return a * b
    def chi_dim(self, degree): return degree
    def chi(self, q, degree): return torch.cat([q**k for k in range(1, degree + 1)], -1)

class Complex(Algebra):
    """ℂ: one complex number per channel. Transition e^{iθ}; lattice: n-th roots of unity (Z_n)."""
    name, gdim, sdim, wdim = "ℂ", 1, 1, 2
    def identity(self, shape, device): return torch.ones(*shape, 1, dtype=torch.cfloat, device=device)
    def _round(self, ang): return torch.round(ang/(2*math.pi/self.lattice))*(2*math.pi/self.lattice) if self.lattice else ang
    def angle(self, g, hard): return self._round(g) if hard else g
    def transition(self, g, hard): return torch.polar(torch.ones_like(g), self.angle(g, hard))
    def write(self, w, hard):
        b = torch.complex(w[..., :1], w[..., 1:])
        if hard: b = torch.polar(b.abs().round().clamp(max=1), self._round(torch.angle(b)))
        return b
    def mul(self, a, b): return a * b
    def scale(self, a, r): return a * r[..., None].to(a.dtype)
    def chi_dim(self, degree): return 2*degree
    def chi(self, q, degree):
        p = [q**k for k in range(1, degree + 1)]
        return torch.cat([torch.cat([z.real, z.imag], -1) for z in p], -1)

class Quaternion(Algebra):
    """ℍ: one quaternion per channel. Transition exp(v), v a pure quaternion (SU(2)); characters q and/or qqᵀ."""
    name, gdim, sdim, wdim = "ℍ", 3, 4, 4
    TRIU = torch.triu_indices(4, 4)
    def __init__(self, lattice=None, sign_invariant=True):
        super().__init__(lattice); self.sign_invariant = sign_invariant
    def identity(self, shape, device):
        e = torch.zeros(*shape, 4, device=device); e[..., 0] = 1; return e
    def transition(self, g, hard):
        n = torch.sqrt((g*g).sum(-1, keepdim=True) + 1e-12)
        return torch.cat([torch.cos(n), torch.sin(n)/n*g], -1)
    def write(self, w, hard): return w
    def mul(self, a, b): return _qmul(a, b)
    def chi_dim(self, degree): return 10 if self.sign_invariant else 14
    def chi(self, q, degree):
        i, j = self.TRIU.to(q.device)
        qq = (q.unsqueeze(-1)*q.unsqueeze(-2))[..., i, j]
        return qq if self.sign_invariant else torch.cat([q, qq], -1)

class MatComplex(Algebra):
    """M_n(ℂ): a dense complex n×n block acting on a vector state h ∈ ℂⁿ (h_0 = e_1). Transitions are general complex
    matrices (not unitary: non-invertible maps such as "× 0" are allowed); characters are the normalized state h/|h|."""
    def __init__(self, n=16, lattice=None): super().__init__(lattice); self.n = n; self.name = f"M{n}(ℂ)"
    def chi_dim(self, degree): return 2*self.n
    def chi(self, q, degree):
        h = q / (q.abs().pow(2).sum(-1, keepdim=True).sqrt() + 1e-30)
        return torch.cat([h.real, h.imag], -1)

class MatrixLayer(nn.Module):
    """One M_n(ℂ) channel: h_t = A(u_t) h_{t-1}, A from a table over the input token (or (previous, current) pair).
    The prefix products are rescaled at every scan step; the readout is scale-invariant, so this is exact."""
    def __init__(self, vocab, alg, conv="pair", init=0.3):
        super().__init__(); self.alg, self.vocab, self.conv, n = alg, vocab, conv, alg.n
        Kt = vocab*vocab if conv == "pair" else vocab
        A = torch.eye(n, dtype=torch.cfloat).repeat(Kt, 1, 1) + init*torch.randn(Kt, n, n, dtype=torch.cfloat)/math.sqrt(n)
        self.A = nn.Parameter(torch.view_as_real(A))
    def forward(self, x, z, hard, st=False):
        tok = (torch.cat([torch.zeros_like(x[:, :1]), x[:, :-1]], 1)*self.vocab + x) if self.conv == "pair" else x
        n = self.alg.n; A = torch.view_as_complex(self.A)[tok]
        A = torch.where((x > 0)[..., None, None], A, torch.eye(n, dtype=torch.cfloat, device=x.device))
        T, d = A.shape[1], 1
        while d < T:
            P = A[:, d:] @ A[:, :-d]
            P = P / (P.abs().pow(2).sum((-1, -2), keepdim=True).sqrt() + 1e-30)
            A = torch.cat([A[:, :d], P], 1); d *= 2
        return A[..., :, 0].unsqueeze(2)                                   # (B, T, 1, n): P_t e_1

ALGEBRAS = {"R": Real, "C": Complex, "H": Quaternion}

class HyperLayer(nn.Module):
    """K channels over one algebra. Tables are indexed by the input token (a linear map on its one-hot code) or by the
    (previous, current) token pair (conv="pair"), and contracted with z = [1, χ(lower layer at t−1)]."""
    def __init__(self, vocab, algebra, K=1, conv="none", couple_dim=0, decay=False, input_term=False):
        super().__init__()
        self.alg, self.K, self.conv, self.vocab = algebra, K, conv, vocab
        self.decay, self.input_term = decay, input_term
        V = vocab*vocab if conv == "pair" else vocab; C = 1 + couple_dim
        g = torch.randn(V, C, K, algebra.gdim)
        if isinstance(algebra, Complex): g[:, 0] = torch.rand(V, K, 1)*2*math.pi        # angles: cover the circle
        if isinstance(algebra, Real): g = g*1.5
        self.E = nn.Parameter(g)
        self.R = nn.Parameter(torch.randn(V, C, K)) if decay else None
        self.W = nn.Parameter(torch.randn(V, C, K, algebra.wdim)*0.5) if input_term else None
    def tokens(self, x):
        if self.conv != "pair": return x
        prev = torch.cat([torch.zeros_like(x[:, :1]), x[:, :-1]], 1)                  # token 0 = no previous token
        return prev*self.vocab + x
    def forward(self, x, z, hard, st=False):
        tok, live = self.tokens(x), (x > 0)
        B, T = x.shape; alg = self.alg
        mix = lambda soft, exact: exact if hard else (soft + (exact - soft).detach() if st else soft)   # straight-through
        g = torch.einsum('btc,btckg->btkg', z, self.E[tok])
        a = mix(alg.transition(g, False), alg.transition(g, True))
        ident = alg.identity((B, T, self.K), x.device)
        a = torch.where(live[..., None, None], a, ident)
        tracking = not (self.decay or self.input_term)
        if tracking and isinstance(alg, Complex):                                     # exact phase accumulation
            ang = mix(g, alg.angle(g, True)) * live[..., None, None]
            return torch.polar(torch.ones_like(ang), torch.cumsum(ang, 1))
        if tracking and isinstance(alg, Real): return torch.cumprod(a, 1)
        if tracking: return prefix_scan(a, alg.mul)
        if self.decay:
            r = torch.sigmoid(torch.einsum('btc,btck->btk', z, self.R[tok])); r = mix(r, r.round())
            a = torch.where(live[..., None, None], alg.scale(a, r), ident)
        if self.input_term:
            w = torch.einsum('btc,btckw->btkw', z, self.W[tok]); b = mix(alg.write(w, False), alg.write(w, True))
            b = b * live[..., None, None].to(b.dtype)
        else:
            b = torch.zeros_like(ident)
        b = torch.cat([alg.mul(a[:, :1], ident[:, :1]) + b[:, :1], b[:, 1:]], 1)    # start from the identity state
        return affine_scan(a, b, alg.mul)

class HyperSSM(nn.Module):
    """A stack of HyperLayers. layers: list of dicts {alg, K, conv, decay, input_term, lattice, couple_degree,
    sign_invariant}; readout: 'top' (χ of the top layer) or 'tensor' (χ_top ⊗ [1, χ_lower])."""
    def __init__(self, vocab, classes, layers, readout="top", readout_degree=1, straight_through=False):
        super().__init__()
        self.specs, self.readout, self.rdeg, self.st = layers, readout, readout_degree, straight_through
        self.algs, mods, prev_alg, prev_K = [], [], None, None
        for spec in layers:
            if spec["alg"] == "M":                                          # dense complex block (no coupling input)
                alg = MatComplex(n=spec.get("n", 16)); mods.append(MatrixLayer(vocab, alg, spec.get("conv", "pair")))
                self.algs.append(alg); prev_alg, prev_K = alg, 1; continue
            kw = {"sign_invariant": spec.get("sign_invariant", True)} if spec["alg"] == "H" else {}
            alg = ALGEBRAS[spec["alg"]](lattice=spec.get("lattice"), **kw)
            cdim = prev_K*prev_alg.chi_dim(spec.get("couple_degree", 1)) if prev_alg is not None else 0
            mods.append(HyperLayer(vocab, alg, spec.get("K", 1), spec.get("conv", "none"), cdim,
                                   spec.get("decay", False), spec.get("input_term", False)))
            self.algs.append(alg); prev_alg, prev_K = alg, spec.get("K", 1)
        self.layers = nn.ModuleList(mods)
        top, K = self.algs[-1], layers[-1].get("K", 1)
        fdim = K*top.chi_dim(readout_degree)
        if readout == "tensor":
            lo, Kl = self.algs[-2], layers[-2].get("K", 1); fdim *= 1 + Kl*lo.chi_dim(1)
        self.head = nn.Linear(fdim, classes)
        self.register_buffer("hard_flag", torch.zeros(()))
    @property
    def hard(self): return bool(self.hard_flag > 0)
    @hard.setter
    def hard(self, v): self.hard_flag.fill_(1.0 if v else 0.0)
    def chi(self, i, q, degree):
        return self.algs[i].chi(q, degree).flatten(2)
    def states(self, x):
        qs, B, T = [], *x.shape
        for i, layer in enumerate(self.layers):
            if i == 0: z = torch.ones(B, T, 1, device=x.device)
            else:
                d = self.specs[i].get("couple_degree", 1)
                c = self.chi(i - 1, qs[-1], d)
                c0 = self.chi(i - 1, self.algs[i - 1].identity((B, 1, self.specs[i - 1].get("K", 1)), x.device), d)
                z = torch.cat([torch.ones(B, T, 1, device=x.device), torch.cat([c0, c[:, :-1]], 1)], -1)  # lower at t−1
            qs.append(layer(x, z, self.hard, self.st))
        return qs
    def forward(self, x):
        qs = self.states(x)
        f = self.chi(len(qs) - 1, qs[-1], self.rdeg)
        if self.readout == "tensor":
            lo = self.chi(len(qs) - 2, qs[-2], 1)
            f = (f.unsqueeze(-1) * torch.cat([torch.ones_like(lo[..., :1]), lo], -1).unsqueeze(-2)).flatten(2)
        return self.head(f)
    @property
    def algebra(self): return " → ".join(f"{a.name}" + (f"×{s.get('K',1)}" if s.get("K", 1) > 1 else "")
                                         for a, s in zip(self.algs, self.specs))

## Presets: the algebra and depth the theory chooses for each task

In [ ]:
def _preset_paper(task):
    """returns (layers, model kwargs, quotient target or None), as in the paper."""
    if task == "parity":     return [dict(alg="R")], {}, None                                   # Z2 (real irrep)
    if task == "cycle_nav":  return [dict(alg="C", lattice=5)], {}, None                        # Z5
    if task == "even_pairs": return [dict(alg="C", lattice=2, conv="pair")], {}, None           # changes between neighbours: Z2
    if task == "mod_arith":                                                                     # dense M16(ℂ) block on token pairs
        return [dict(alg="M", n=16, conv="pair")], {}, None                                     #   (learnable; see the cascade below)
    if task == "mod_arith_cascade":                                                             # exact 3-number Krohn–Rhodes cascade
        return [dict(alg="C", lattice=4, conv="pair", decay=True, input_term=True),             #   product: Z4 ∪ {0}, reset per term
                dict(alg="C", lattice=5, couple_degree=4)], {"straight_through": True}, None    #   sum: Z5, reads χ(product)
    if task.startswith("A5"): return [dict(alg="H")], {}, None                                  # A5 = 2I/±1, sign-invariant χ
    if task.startswith("S5"):                                                                   # S5: sign below A5 (Prop. 2)
        return [dict(alg="R"), dict(alg="H")], {"readout": "tensor"}, "sign"
    raise ValueError(task)

def preset(task):
    """The paper's preset with the task-specific choices removed: no ℂ lattice is given (it is discovered after
    training), ℝ channels are ±1 straight-through, and the S5 sign layer is not pretrained."""
    layers, kw, quotient = _preset_paper(task)
    if task == "mod_arith_cascade":                     # the hand-set representation check is left unchanged
        return layers, kw, quotient
    layers = [dict(l) for l in layers]
    if DISCOVER_LATTICE:
        for l in layers:
            if l["alg"] == "C": l["lattice"] = None
    if REAL_STRAIGHT_THROUGH and any(l["alg"] == "R" for l in layers):
        kw = dict(kw, straight_through=True)
    if not QUOTIENT_FIRST:
        quotient = None
    return layers, kw, quotient

def build(task, vocab, classes):
    layers, kw, _ = preset(task); return HyperSSM(vocab, classes, layers, **kw)

## Data and evaluation helpers

In [ ]:
def final_logits(model, x, L):
    out = model(x)
    return out[torch.arange(x.shape[0], device=x.device), L - 1]

@torch.no_grad()
def evaluate(model, data, bs=64):
    model.eval(); correct = 0
    xs, ys, Ls = data
    for i in range(0, len(xs), bs):
        x = torch.as_tensor(xs[i:i+bs], device=device); L = torch.as_tensor(Ls[i:i+bs], device=device)
        correct += (final_logits(model, x, L).argmax(-1).cpu().numpy() == ys[i:i+bs]).sum()
    model.train(); return correct / len(xs)

def batches(gen, n, lmin, lmax, seed, bs=256):
    rng = np.random.default_rng(seed); xs, ys, Ls = [], [], []
    for i in range(0, n, bs):
        x, y, L = gen(min(bs, n - i), lmin, lmax, rng)
        x = np.pad(x, ((0, 0), (0, lmax + 1 - x.shape[1])))[:, :lmax + 1]
        xs.append(x); ys.append(y); Ls.append(L)
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(Ls)

## Representation checks (no training)
1. The quaternion algebra composes exactly like SU(2) matrices.
2. **Modular arithmetic is exactly representable** by the 2-layer cascade preset: we set its parameters by hand
   (layer A: product in Z4 ∪ {0} with a reset per term; layer B: sum in Z5, whose turn is a *linear* function of the
   characters of layer A, solved by least squares on the 5 group states) and check it at lengths up to 1000.

In [ ]:
with torch.no_grad():                                           # i ↔ −iσx, j ↔ −iσy, k ↔ −iσz
    H = Quaternion(); g = torch.randn(64, 3).double()*2
    SIG = [torch.tensor([[0, 1], [1, 0]], dtype=torch.cdouble), torch.tensor([[0, -1j], [1j, 0]], dtype=torch.cdouble),
           torch.tensor([[1, 0], [0, -1]], dtype=torch.cdouble)]
    M = torch.eye(2, dtype=torch.cdouble)
    for v in g: M = torch.linalg.matrix_exp(-1j*sum(v[k]*SIG[k] for k in range(3))) @ M
    w, x, y, z = prefix_scan(H.transition(g[None, :, None, :], False), H.mul)[0, -1, 0]
    Mq = torch.tensor([[w - 1j*z, -1j*x - y], [-1j*x + y, w + 1j*z]], dtype=torch.cdouble)
    print(f"quaternion scan vs product of SU(2) matrices (64 steps): max |difference| = {(Mq - M).abs().max().item():.2e}")

gen_m, vocab_m, classes_m, _ = task_spec("mod_arith")
m = build("mod_arith_cascade", vocab_m, classes_m)
LOG = {1: 0, 2: 1, 4: 2, 3: 3}; BIG = 30.0
with torch.no_grad():
    A, Bl = m.layers
    A.E.zero_(); A.R.fill_(BIG); A.W.zero_()
    for p in range(vocab_m):
        for c in range(5, 10):
            v, i = c - 5, p*vocab_m + c
            if p in (0, 1, 2):                                  # new term (start, after '+', after '−'): reset, write ±v
                A.R[i] = -BIG
                if v: k = LOG[v] + (2 if p == 2 else 0); A.W[i, 0, 0] = torch.tensor([math.cos(k*math.pi/2), math.sin(k*math.pi/2)])
            elif p == 3:                                        # after '×': multiply (or reset to 0 for ×0)
                if v: A.E[i, 0, 0, 0] = LOG[v]*math.pi/2
                else: A.R[i] = -BIG
    states = [torch.zeros(1, dtype=torch.cfloat)] + [torch.polar(torch.ones(1), torch.tensor([k*math.pi/2])) for k in range(4)]
    target = [0.0] + [2*math.pi*(2**k % 5)/5 for k in range(4)]
    Z = torch.stack([torch.cat([torch.ones(1), Complex().chi(s[None], 4)[0]]) for s in states]).double()
    coef = torch.linalg.lstsq(Z, torch.tensor(target, dtype=torch.double)[:, None]).solution[:, 0].float()
    Bl.E.zero_()
    for c in (1, 2, 4): Bl.E[c, :, 0, 0] = coef                 # '+', '−', '=' fold the product into the sum
    m.head.weight.zero_(); m.head.bias.fill_(-10.0)
    for j in range(5):
        m.head.weight[5 + j] = torch.tensor([math.cos(2*math.pi*j/5), math.sin(2*math.pi*j/5)])*10; m.head.bias[5 + j] = 0
m = m.to(device); m.hard = True
for lo, hi in [(3, 40), (40, 256), (256, 1000)]:
    print(f"hand-set cascade ({m.algebra}), lengths {lo}–{hi}: accuracy {evaluate(m, batches(gen_m, 256, lo, hi, 7 + hi)):.4f}")

## Training toolkit
* **Short-length curriculum hold**: lengths 3–5 for the first `CURR_HOLD` of training (near-direct supervision of how
  generators compose, a smooth loss landscape), then a ramp to 40. Training lengths always stay within 3–40.
* **Snapped model selection**: every checkpoint is scored in hard mode (exact lattice), on lengths 40–256.
* **Restarts**: an attempt still below `ABORT_BELOW` after `ABORT_AT` of training is in the wrong basin and is
  restarted from a fresh initialization (the number of attempts is recorded).
* **Quotient first** (S5, only if `QUOTIENT_FIRST = True`): the bottom ℝ± layer is trained alone on the sign of the
  label, snapped to ±1 and frozen. Off by default: the layers are trained jointly.
* **Lattice discovery** (if `DISCOVER_LATTICE`): each complex layer is snapped to the smallest n (≤ `LATTICE_MAX`)
  whose roots of unity give the best snapped accuracy on lengths 40–256, the set already used for checkpoint
  selection. No n is given per task. During training this snapped score drives checkpoint selection, early stopping
  and restarts (training itself stays continuous); after training, n is fixed for the best checkpoint.
* **Snapped fine-tune**: `FINETUNE_STEPS` of training with the discovered lattice in the forward pass: the snapped
  complex layers are held fixed and the rest (the readout) is trained on the snapped states, keeping the best snapped
  checkpoint, so the model evaluated is the model last trained.

In [ ]:
def quotient_stage(model, gen, info, seed, steps=1500, bs=256, restarts=8):
    sign = torch.tensor(info["sign"], dtype=torch.float32, device=device); layer = model.layers[0]
    hx, hy, hL = (torch.as_tensor(a, device=device) for a in gen(1024, 3, 40, np.random.default_rng(seed + 999)))
    def final_state(x, L, hard):
        q = layer(x, torch.ones(*x.shape, 1, device=device), hard)
        return q[torch.arange(len(L), device=device), L - 1, 0, 0]
    best_err, best = float("inf"), None
    for r in range(restarts):
        rng = np.random.default_rng(seed + 777 + r)
        with torch.no_grad(): layer.E.copy_(torch.randn_like(layer.E)*1.5)
        opt = torch.optim.Adam([layer.E], lr=3e-2)
        for it in range(steps):
            x, y, L = (torch.as_tensor(a, device=device) for a in gen(bs, 3, 3 + it*8//steps, rng))   # lengths 3–10
            loss = ((final_state(x, L, False) - sign[y])**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): err = ((final_state(hx, hL, True) - sign[hy])**2).mean().item()
        if err < best_err: best_err, best = err, layer.E.detach().clone()
        if best_err < 1e-6: break
    with torch.no_grad(): layer.E.copy_(torch.sign(best)*20)          # exactly ±1 (tanh(20) = 1 in float32)
    layer.E.requires_grad_(False)
    return best_err

def discover_lattices(model, val, log=True):
    """For each complex layer, the smallest n ≤ LATTICE_MAX whose n-th roots of unity give the best snapped accuracy
    on the validation set. Uses only the trained model and the validation data. Leaves the lattices set."""
    found = {}
    for i, alg in enumerate(model.algs):
        if not isinstance(alg, Complex) or alg.lattice is not None:
            continue
        best_n, best_acc = None, -1.0
        for n in range(1, LATTICE_MAX + 1):
            alg.lattice = n; model.hard = True
            acc = evaluate(model, val); model.hard = False
            if acc > best_acc + 1e-9:
                best_n, best_acc = n, acc
            if best_acc >= 0.9995:
                break                                        # the smallest exact n
        alg.lattice = best_n; found[i] = best_n
        if log: print(f"    layer {i} (ℂ): discovered n = {best_n}, snapped val {best_acc:.4f}", flush=True)
    return found

def selection_score(model, val):
    """Validation accuracy of the snapped model, used for checkpoint selection, early stopping and restarts.
    ℝ is ±1 and ℍ continuous as always; complex layers without a lattice are snapped to their discovered n for this
    evaluation only (training itself stays continuous)."""
    found = discover_lattices(model, val, log=False) if DISCOVER_LATTICE else {}
    model.hard = True; acc = evaluate(model, val); model.hard = False
    for i in found: model.algs[i].lattice = None
    return acc

def finetune_snapped(model, gen, val, task, seed, log=True):
    """Train briefly with the discovered lattice in the forward pass; keep the best snapped checkpoint on lengths
    40–256. The snapped complex layers are held fixed (their transitions are now exact roots of unity) and the rest of
    the model, such as the readout, is trained on the snapped states. (Letting Adam move the raw phases under a
    straight-through estimator makes them random-walk across rounding boundaries, since their gradients are ~0.)"""
    model.st = True
    for i, alg in enumerate(model.algs):
        if isinstance(alg, Complex) and alg.lattice is not None:
            for p in model.layers[i].parameters(): p.requires_grad_(False)
    params = [p for p in model.parameters() if p.requires_grad]
    lr = FINETUNE_LR_FACTOR*LR_BY_TASK.get(task, LR)
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    rng = np.random.default_rng(seed + 4242)
    model.hard = True; best = evaluate(model, val); model.hard = False
    state = copy.deepcopy(model.state_dict())
    for s in range(1, FINETUNE_STEPS + 1):            # always runs, so the final model is trained in snapped form
        x, y, L = (torch.as_tensor(a, device=device) for a in gen(BATCH, 3, 40, rng))
        loss = F.cross_entropy(final_logits(model, x, L), y)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step()
        if s % EVAL_EVERY == 0 or s == FINETUNE_STEPS:
            model.hard = True; acc = evaluate(model, val); model.hard = False
            if acc >= best: best, state = acc, copy.deepcopy(model.state_dict())   # ties: keep the fine-tuned state
            if log: print(f"    fine-tune step {s:5d} loss {loss.item():.3f} val(40–256, snapped) {acc:.4f} best {best:.4f}",
                          flush=True)
    model.load_state_dict(state)
    return best

def run_one(task, seed, steps=STEPS, log=True):
    gen, vocab, classes, info = task_spec(task)
    _, _, quotient = preset(task)
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = build(task, vocab, classes).to(device)
    if quotient == "sign": quotient_stage(model, gen, info, seed)
    frozen = copy.deepcopy(model.layers[0].state_dict()) if quotient else None
    val = batches(gen, N_VAL, 40, 256, 10_000 + seed)
    ext = batches(gen, N_EXT, 256, 1000, 20_000 + seed)
    val_sel = tuple(a[:N_SEL] for a in val)        # subset used during training (the lattice search runs at every eval)
    n_attempts = 1 + (RESTARTS_ABELIAN if task in ("parity", "cycle_nav", "even_pairs") else
                      RESTARTS_MOD if task == "mod_arith" else RESTARTS)
    best, best_state, t0 = -1.0, None, time.time()
    for attempt in range(n_attempts):
        if attempt:
            torch.manual_seed(1000*seed + attempt); model = build(task, vocab, classes).to(device)
            if frozen is not None:
                model.layers[0].load_state_dict(frozen); model.layers[0].E.requires_grad_(False)
        params = [p for p in model.parameters() if p.requires_grad]
        warm = max(1, int(0.1*steps))
        lr = LR_BY_TASK.get(task, LR)
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s < warm else
                                                  (1e-5 + 0.5*(lr - 1e-5)*(1 + math.cos(math.pi*(s - warm)/max(1, steps - warm))))/lr)
        rng = np.random.default_rng(seed + 31*attempt); a_best, a_state = -1.0, None
        for s in range(1, steps + 1):
            frac = s/steps
            lmax = 3 + int(2*min(1, frac/CURR_HOLD)) if frac < CURR_HOLD else min(40, 5 + int((frac - CURR_HOLD)/0.4*35))
            x, y, L = (torch.as_tensor(a, device=device) for a in gen(BATCH, 3, lmax, rng))
            loss = F.cross_entropy(final_logits(model, x, L), y)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()
            if s % EVAL_EVERY == 0 or s == steps:
                acc = selection_score(model, val_sel)
                if acc > a_best: a_best, a_state = acc, copy.deepcopy(model.state_dict())
                if log: print(f"    {task:10s} seed {seed} try {attempt} step {s:6d} loss {loss.item():.3f} "
                              f"val(40–256, snapped) {acc:.4f} best {a_best:.4f} [{time.time()-t0:.0f}s]", flush=True)
                if EARLY_STOP and a_best >= 0.9995: break
                if attempt < n_attempts - 1 and s >= ABORT_AT*steps and a_best < ABORT_BELOW: break
        if a_best > best: best, best_state, best_model = a_best, a_state, model
        if best >= 0.9995: break
    model = best_model; model.load_state_dict(best_state); model.hard = True
    val_cont, ext_cont = float(evaluate(model, val)), float(evaluate(model, ext))   # ℂ continuous, ℝ ±1, ℍ continuous
    model.hard = False
    lattices = discover_lattices(model, val, log) if DISCOVER_LATTICE else {}
    if lattices:
        best = finetune_snapped(model, gen, val, task, seed, log)
    model.hard = True
    return dict(task=task, model="HyperSSM", seed=seed, val=float(best), ext=float(evaluate(model, ext)),
                val_continuous=val_cont, ext_continuous=ext_cont,
                lattices={str(k): v for k, v in lattices.items()},
                snapping="task-free" if (DISCOVER_LATTICE and REAL_STRAIGHT_THROUGH and not QUOTIENT_FIRST) else "paper",
                supervision="final", attempts=attempt + 1, secs=time.time() - t0,
                params=sum(p.numel() for p in model.parameters()), algebra=model.algebra)

## Run (resumable)

In [ ]:
MODELS = ["HyperSSM"]
done = set()
if os.path.exists(RESULTS_PATH):
    for line in open(RESULTS_PATH):
        r = json.loads(line); done.add((r["task"], r["seed"]))
for task in TASKS:
    for seed in SEEDS:
        if (task, seed) in done: continue
        r = run_one(task, seed)
        lat = ", ".join(f"layer {k}: n={v}" for k, v in r["lattices"].items()) or "none"
        print(f"==> {task:10s} seed {seed}  [{r['algebra']}]  continuous: val {r['val_continuous']:.4f} "
              f"ext {r['ext_continuous']:.4f} | discovered {lat} | final: val(40–256) {r['val']:.4f} "
              f"ext(256–1000) {r['ext']:.4f}  attempts {r['attempts']}  ({r['secs']:.0f}s)", flush=True)
        with open(RESULTS_PATH, "a") as f: f.write(json.dumps(r) + "\n")

## Results: best of seeds (the paper's metric), with the number of seeds above 99% in brackets

In [ ]:
rows = [json.loads(l) for l in open(RESULTS_PATH)]
tasks = [t for t in TASKS if any(r["task"] == t for r in rows)]
models = [m for m in MODELS if any(r["model"] == m for r in rows)]
def cell(m, t, key):
    rs = [r for r in rows if r["model"] == m and r["task"] == t]
    if not rs: return np.nan, ""
    a = np.array([r[key] for r in rs]); return a.max(), f"{a.max()*100:.1f}\n({(a > 0.99).sum()}/{len(a)})"
LABEL = {"parity": "Parity", "even_pairs": "Even pairs", "cycle_nav": "Cycle nav", "mod_arith": "Mod arith"}
xt = [LABEL.get(t, t.replace("_", ", ")) for t in tasks]
fig, axes = plt.subplots(2, 1, figsize=(1.05*len(tasks) + 2.5, 1.0*len(models)*2 + 2))
for ax, key, title in [(axes[0], "val", "lengths 40–256 (PD-SSM protocol)"), (axes[1], "ext", "lengths 256–1000 (extension)")]:
    M = np.array([[cell(m, t, key)[0] for t in tasks] for m in models])
    ax.imshow(M, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    for i, m in enumerate(models):
        for j, t in enumerate(tasks):
            v, s = cell(m, t, key)
            if s: ax.text(j, i, s, ha="center", va="center", fontsize=7, color="w" if v < 0.6 else "k")
    alg = {m: "" for m in models}
    hy = [r for r in rows if r["model"] == "HyperSSM"]
    ax.set_yticks(range(len(models))); ax.set_yticklabels(models, fontsize=9)
    ax.set_xticks(range(len(tasks))); ax.set_xticklabels(xt, fontsize=8, rotation=30, ha="right")
    ax.set_title(f"Best-of-seeds accuracy, {title}", fontsize=10)
    for j, t in enumerate(tasks):
        a = next((r["algebra"] for r in hy if r["task"] == t), "")
        if a and "HyperSSM" in models: ax.text(j, models.index("HyperSSM") + 0.38, a, ha="center", va="center", fontsize=6.5, color="w")
plt.tight_layout(); plt.savefig("hyperssm_fsm.png", dpi=150); plt.show()

REF = {   # Terzić et al. (NeurIPS 2025), Table 3: best validation accuracy (lengths up to 256), best of 3 seeds, 100k steps.
         # Transcribed by hand: verify against the paper before using. "-" = not reported.
    "PD-SSM (1 layer)":        {"A5_2": 100.0, "A5_6": 100.0, "A5_8": 100.0, "A5_12": 100.0, "S5_4": 100.0, "S5_8": 100.0, "S5_32": 1.07},
    "Gated DeltaProduct (1)":  {"A5_2": 97.9, "A5_6": 92.5, "A5_8": 91.8, "A5_12": 60.5, "S5_4": 88.7, "S5_8": 57.4, "S5_32": 1.23},
    "Gated DeltaProduct (2)":  {"A5_12": 68.4, "S5_4": 84.7, "S5_8": 62.2},
    "ℂ diagonal (2)":          {"A5_2": 15.5},
}
FSA = ["parity", "even_pairs", "cycle_nav", "mod_arith"]
REF2 = {  # Terzić et al. (NeurIPS 2025), Table 2: mean ± std validation accuracy over 5 seeds. Transcribed: verify.
    "PD-SSM":                   {"cycle_nav": "99.5±0.7", "even_pairs": "99.7±0.3", "mod_arith": "96.2±3.4", "parity": "99.9±0.1"},
    "Gated DeltaProduct[-1,1]": {"cycle_nav": "61.4±12.0", "even_pairs": "99.6±0.6", "mod_arith": "58.0±20.0", "parity": "98.8±1.5"},
    "DeltaNet[-1,1]":           {"cycle_nav": "47.3±6.0", "even_pairs": "100.0±0.0", "mod_arith": "63.9±7.2", "parity": "97.2±1.7"},
    "ℂ Diag.":                  {"cycle_nav": "90.4±4.3", "even_pairs": "82.4±4.8", "mod_arith": "59.9±27.0", "parity": "61.8±6.8"},
    "Mamba":                    {"cycle_nav": "50.7±7.8", "even_pairs": "100.0±0.0", "mod_arith": "38.1±6.0", "parity": "54.5±1.5"},
    "Transformer":              {"cycle_nav": "24.4±0.5", "even_pairs": "90.4±10.4", "mod_arith": "23.6±0.7", "parity": "52.2±0.4"},
    "LSTM":                     {"cycle_nav": "100.0±0.0", "even_pairs": "100.0±0.0", "mod_arith": "99.9±0.1", "parity": "100.0±0.0"},
}
ft = [t for t in FSA if t in tasks]
if ft:
    print("\nTable 2 protocol (FSA): mean ± std over seeds, lengths 40–256")
    print(f"{'model':26s}" + "".join(f"{LABEL[t]:>14s}" for t in ft))
    for name, d in REF2.items(): print(f"{name:26s}" + "".join(f"{d[t]:>14s}" for t in ft))
    for m in models:
        cells = []
        for t in ft:
            a = np.array([r["val"] for r in rows if r["model"] == m and r["task"] == t])*100
            cells.append(f"{a.mean():.1f}±{a.std():.1f}" if len(a) else "-")
        print(f"{m + ' (ours)':26s}" + "".join(f"{c:>14s}" for c in cells))
print("\nTable 3 protocol (A5/S5): best of seeds")
print("\nNote: their A5/S5 extra generators are random and not public, so the tasks match in size, not exactly.")
print(f"\n{'model':24s}" + "".join(f"{x:>11s}" for x in xt))
for name, d in REF.items():
    print(f"{name:24s}" + "".join(f"{(str(d[t]) if t in d else '-'):>11s}" for t in tasks))
for m in models:
    def mark(t): return "*" if any(r["model"] == m and r["task"] == t and r.get("supervision") == "all" for r in rows) else ""
    val_cells = [f"{cell(m, t, 'val')[0]*100:.1f}{mark(t)}" for t in tasks]
    ext_cells = [f"{cell(m, t, 'ext')[0]*100:.1f}" for t in tasks]
    print(f"{m + ' (ours)':24s}" + "".join(f"{c:>11s}" for c in val_cells))
    print(f"{'  at lengths 256–1000':24s}" + "".join(f"{c:>11s}" for c in ext_cells))

print("\nTask-free snapping: continuous vs final, and the discovered ℂ lattices (all seeds)")
print(f"{'task':12s}{'val cont.':>11s}{'ext cont.':>11s}{'val final':>11s}{'ext final':>11s}   discovered n (per seed)")
for t in tasks:
    rs = [r for r in rows if r["task"] == t and "val_continuous" in r]
    if not rs: continue
    f = lambda k: f"{100*np.mean([r[k] for r in rs]):.1f}"
    lat = [",".join(str(v) for v in r["lattices"].values()) or "–" for r in rs]
    print(f"{t:12s}{f('val_continuous'):>11s}{f('ext_continuous'):>11s}{f('val'):>11s}{f('ext'):>11s}   {' '.join(lat)}")
print("(means over seeds; attempts per seed are in the results file)")